In [8]:
from pathlib import Path
from dotenv import load_dotenv
import os
import requests
import json
import pandas as pd
import yfinance as yf

# Find project root (works from root or notebooks/)
project_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / ".env").exists()
)

load_dotenv(project_root / ".env")

API_KEY = os.getenv("ALPHA_VINTAGE_API_KEY")

if not API_KEY:
    raise ValueError("ALPHA_VINTAGE_API_KEY not found in .env")

RAW_DIR = project_root / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("API key loaded successfully")

Project root: /Users/nikolaskefalonitis/Desktop/All_folders/projects/quant/btcVSsp500
API key loaded successfully


In [5]:
url = "https://www.alphavantage.co/query"

params = {
    "function": "DIGITAL_CURRENCY_DAILY",
    "symbol": "BTC",
    "market": "USD",
    "apikey": API_KEY
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

data = response.json()

series_key = "Time Series (Digital Currency Daily)"

if series_key not in data:
    raise RuntimeError(
        f"Alpha Vantage did not return BTC data: {data}"
    )

# Save original API response
output_path = RAW_DIR / "btc_usd_daily.json"

with open(output_path, "w") as f:
    json.dump(data, f, indent=2)

print(f"Saved BTC data to: {output_path}")
print(f"Records: {len(data[series_key])}")
print(f"Earliest: {min(data[series_key])}")
print(f"Latest: {max(data[series_key])}")

Saved BTC data to: /Users/nikolaskefalonitis/Desktop/All_folders/projects/quant/btcVSsp500/data/raw/btc_usd_daily.json
Records: 5963
Earliest: 2010-07-17
Latest: 2026-11-12


In [6]:
btc = pd.DataFrame.from_dict(
    data[series_key],
    orient="index"
)

btc.index = pd.to_datetime(btc.index)
btc.index.name = "date"

btc = btc.sort_index()

# Convert numeric columns
btc = btc.apply(pd.to_numeric, errors="coerce")

display(btc.head())
print(btc.columns.tolist())

,1. open,2. high,3. low,4. close,5. volume
date,,,,,
2010-07-17,0.0500,0.0500,0.0500,0.0500,0.0
2010-07-18,0.0858,0.0858,0.0858,0.0858,0.0
2010-07-19,0.0808,0.0808,0.0808,0.0808,0.0
2010-07-20,0.0747,0.0747,0.0747,0.0747,0.0
2010-07-21,0.0792,0.0792,0.0792,0.0792,0.0


['1. open', '2. high', '3. low', '4. close', '5. volume']


In [7]:
PROCESSED_DIR = project_root / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

btc.to_csv(PROCESSED_DIR / "btc_usd_daily.csv")

print("Processed BTC dataset saved.")

Processed BTC dataset saved.


In [9]:
spy = yf.download(
    "SPY",
    start="1993-01-01",
    interval="1d",
    auto_adjust=True,
    multi_level_index=False,
    progress=False
)

if spy.empty:
    raise RuntimeError("Yahoo Finance returned no SPY data")

# Keep only adjusted closing prices
spy = spy[["Close"]].copy()
spy.columns = ["adjusted_close"]
spy.index.name = "date"

# Save raw data
raw_path = RAW_DIR / "spy_usd_daily.csv"
spy.to_csv(raw_path)

print(f"Saved SPY data to: {raw_path}")
print(f"Records: {len(spy)}")
print(f"Earliest: {spy.index.min()}")
print(f"Latest: {spy.index.max()}")

Saved SPY data to: /Users/nikolaskefalonitis/Desktop/All_folders/projects/quant/btcVSsp500/data/raw/spy_usd_daily.csv
Records: 8506
Earliest: 1993-01-29 00:00:00
Latest: 2026-11-12 00:00:00


In [10]:
spy_processed = spy.sort_index().dropna()

PROCESSED_DIR = project_root / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

processed_path = PROCESSED_DIR / "spy_usd_daily.csv"
spy_processed.to_csv(processed_path)

print(f"Processed SPY data saved to: {processed_path}")

display(spy_processed.head())
display(spy_processed.tail())

Processed SPY data saved to: /Users/nikolaskefalonitis/Desktop/All_folders/projects/quant/btcVSsp500/data/processed/spy_usd_daily.csv


,adjusted_close
date,
1993-01-29,24.053532
1993-02-01,24.224623
1993-02-02,24.275940
1993-02-03,24.532526
1993-02-04,24.635183


,adjusted_close
date,
2026-11-06,735.289978
2026-11-09,752.690002
2026-11-10,749.919983
2026-11-11,759.809998
2026-11-12,759.135010
